<a href="https://colab.research.google.com/github/muriloous/ied003/blob/main/atv3/estruturadedados_atv3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Atividade aula 4 - 12 de março
**Murilo Martho - 1141352613002**

**Enunciado:**

A NASA disponibiliza a API Near Earth Object Web Service (NEO), que fornece dados sobre objetos espaciais (asteroides e cometas) que passam próximos à Terra.
Sua tarefa é desenvolver um script em Python que:
- Pergunte ao usuário uma data inicial. O formato que a API aceita é YYYY-MM-DD.
- Pergunte ao usuário uma data final.

Dica: caso o usuário não informe a data final, considere como padrão 7 dias após a data inicial (dica: use a biblioteca datetime para calcular isso).

- Acesse a API da NASA para obter os objetos próximos à Terra no período especificado.

- Organize os dados em um dicionário e exiba para cada objeto:
  - Nome do objeto espacial
  - Diâmetro máximo em metros
  - Distância mínima da Terra (em km)
  - Se é potencialmente perigoso (Sim - True ou Não - False)
  - Corpo celeste que orbita (exemplo: 'Earth')

In [ ]:
import requests
import sys
from datetime import date, timedelta, datetime
from pprint import pprint

def build_url(start_date = None, end_date = None):
  # Chave da API hardcoded para fins dessa atividade
  API_KEY = 'DEMO_KEY'
  start_date = start_date if start_date else date.today() - timedelta(days=7)
  end_date = end_date if end_date else start_date + timedelta(days=7)

  if start_date > end_date:
    print('WARNING: Data final é anterior à data inicial. Ajustando data final para 7 dias após a data inicial.')
    end_date = start_date + timedelta(days=7)

  elif (end_date - start_date).days > 7:
    print('WARNING: O intervalo entre a data inicial e final excede 7 dias. Ajustando a data final para 7 dias após a data inicial.')
    end_date = start_date + timedelta(days=7)

  return f'https://api.nasa.gov/neo/rest/v1/feed?start_date={start_date}&end_date={end_date}&api_key={API_KEY}'

def is_valid_date(date, warns = True):
  chuncks = date.split('-')
  if not len(chuncks) == 3 and all(p.isdigit() for p in chuncks):
    if warns: print(f"Data inválida: {date}")
    return False
  return True

def parse_dict(url):
  res = requests.get(url)

  if res.status_code == 200:
    res_json = res.json()
    near_earth_objects = res_json.get('near_earth_objects', {})

    if not near_earth_objects:
      print("Nenhum objeto próximo à Terra encontrado para o período especificado.")
      return

    print("\n--- Objetos próximos à Terra ---")
    parsed_dict = {}
    for date_key, objects_list in near_earth_objects.items():
      parsed_dict[date_key] = []
      for obj in objects_list:
        parsed_dict[date_key].append({
          'Nome': obj.get('name', 'N/A'),
          'Diâmetro máximo (metros)': obj.get('estimated_diameter', {}).get('meters', {}).get('estimated_diameter_max', 'N/A'),
          'Distância mínima da terra (km)': obj['close_approach_data'][0].get('miss_distance', {}).get('kilometers', 'N/A'),
          'Potencialmente perigoso': obj.get('is_potentially_hazardous_asteroid', False),
          'Corpo Celeste que Orbita': obj['close_approach_data'][0].get('orbiting_body', 'N/A')
        })
    return parsed_dict

  else:
    print(f"Erro ao acessar a API da NASA: {res.status_code} - {res.text}")

def main():
  raw_start_date = input("Digite a data inicial no formato YYYY-MM-DD: ")
  start_date = datetime.strptime(raw_start_date, '%Y-%m-%d').date() if is_valid_date(raw_start_date) else None

  raw_end_date = input("Digite a data final no formato YYYY-MM-DD (deixe em branco para 7 dias após a data inicial): ")
  end_date = datetime.strptime(raw_end_date, '%Y-%m-%d').date() if is_valid_date(raw_end_date) else None

  url = build_url(start_date, end_date)
  print(f"URL a ser consultada: {url}")

  parsed = parse_dict(url)

  pprint(parsed)

if __name__ == "__main__":
  main()

Digite a data inicial no formato YYYY-MM-DD: 2026-03-13
Digite a data final no formato YYYY-MM-DD (deixe em branco para 7 dias após a data inicial): 2026-03-19
URL a ser consultada: https://api.nasa.gov/neo/rest/v1/feed?start_date=2026-03-13&end_date=2026-03-19&api_key=DEMO_KEY

--- Objetos próximos à Terra ---
{'2026-03-13': [{'Corpo Celeste que Orbita': 'Earth',
                 'Distância mínima da terra (km)': '64389321.017620147',
                 'Diâmetro máximo (metros)': 23.6613750114,
                 'Nome': '(2010 JA)',
                 'Potencialmente perigoso': False},
                {'Corpo Celeste que Orbita': 'Earth',
                 'Distância mínima da terra (km)': '71096934.557553998',
                 'Diâmetro máximo (metros)': 149.2931834393,
                 'Nome': '(2011 ON24)',
                 'Potencialmente perigoso': False},
                {'Corpo Celeste que Orbita': 'Earth',
                 'Distância mínima da terra (km)': '10497458.794110434',
   